# This notebook is for making figure 1 of the paper.
- A). Schematic of the model (No plots in this notebook)
- B). KEGG of new metabolic genes
- C). Distribution of new reaction flux to show that they are just low flux reactions. --> indicating the problem
- D). Problem solution

In [4]:
import requests
import pandas as pd
from pathlib import Path
import ast

## B). KEGG of new metabolic genes

In [5]:
outdir = Path("out/new_genes_functionality")
outdir.mkdir(exist_ok=True)

tmpdir = Path("tmp")
tmpdir.mkdir(exist_ok=True)

csv_path = Path("new_metabolic_gene_annotation.csv")
df = pd.read_csv(csv_path)

for col in ["Reactions", "MultiFuntional name"]:
    df[col] = df[col].apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str) and x.strip().startswith("[")
        else x
    )

study_ecocyc_ids  = set(df["Gene ID (EcoCyc)"].dropna().str.strip())
study_locus_ids   = set(df["Gene locus ID"].dropna().str.strip())
study_gene_names  = set(df["Gene name"].dropna().str.strip().str.lower())

print(f"Loaded {len(df)} gene annotations")
print(f"  EcoCyc IDs : {len(study_ecocyc_ids)}")
print(f"  Locus IDs  : {len(study_locus_ids)}")
print(f"  Gene names : {len(study_gene_names)}")
df.head(3)

Loaded 307 gene annotations
  EcoCyc IDs : 307
  Locus IDs  : 307
  Gene names : 307


,Gene ID (EcoCyc),Gene locus ID,Gene name,Enzyme encoded,Pathways,Pathways parent,Protein products,MultiFuntional ID,MultiFuntional name,Reactions,Description by Cyrus
0,EG10022,b4015,aceA,ISOCIT-LYASE,GLYOXYLATE-BYPASS,Energy-Metabolism,ISOCIT-LYASE-MONOMER,BC-1.7.2,[metabolism -> central intermediary metabolism...,[ISOCIT-CLEAV-RXN],acetate transport + metabolism
1,EG10023,b4014,aceB,MALATE-SYNTHASE,GLYOXYLATE-BYPASS,Energy-Metabolism,MALATE-SYNTHASE,BC-1.7.2,[metabolism -> central intermediary metabolism...,[MALSYN-RXN],acetate transport + metabolism
2,EG11942,b4067,actP,CPLX0-7955,NaN,NaN,YJCG-MONOMER,"['BC-4.2.A', 'BC-6.1']",[transport -> Electrochemical potential driven...,"[RXN0-1981, RXN0-5111, TRANS-RXN0-576]",acetate transport + metabolism


In [6]:
kegg_link_path = tmpdir / "kegg_eco_pathway_genes.tsv"
kegg_list_path = tmpdir / "kegg_eco_pathway_names.tsv"

def _kegg_get(url, cache_path, label):
    if not cache_path.exists():
        print(f"Downloading {label}...")
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            cache_path.write_text(r.text, encoding="utf-8")
            print(f"  {len(r.text.splitlines())} entries saved to {cache_path}")
            return cache_path
        except Exception as e:
            print(f"  WARNING: {e}")
            return None
    else:
        print(f"Using cached {label}: {cache_path}")
        return cache_path

kegg_link_path = _kegg_get(
    "https://rest.kegg.jp/link/eco/pathway", kegg_link_path, "KEGG pathway-gene links"
)
kegg_list_path = _kegg_get(
    "https://rest.kegg.jp/list/pathway/eco", kegg_list_path, "KEGG pathway names"
)

kegg_link_df    = pd.DataFrame()
pathway_name_map = {}
kegg_study_genes = set()
kegg_background  = set()

if kegg_link_path is not None:
    kegg_link_df = pd.read_csv(
        kegg_link_path, sep="\t", header=None, names=["pathway_id", "gene_id"]
    )
    # gene_id format: eco:b4015 → strip prefix
    kegg_link_df["locus_id"]      = kegg_link_df["gene_id"].str.replace("eco:", "", regex=False)
    kegg_link_df["pathway_short"] = kegg_link_df["pathway_id"].str.replace("path:", "", regex=False)
    kegg_background = set(kegg_link_df["locus_id"].unique())
    kegg_study_genes = study_locus_ids & kegg_background
    print(f"KEGG: {kegg_link_df['pathway_short'].nunique()} pathways, "
          f"{len(kegg_background)} background genes")
    print(f"Study genes matched: {len(kegg_study_genes)} / {len(study_locus_ids)}")

if kegg_list_path is not None:
    names_df = pd.read_csv(kegg_list_path, sep="\t", header=None, names=["pathway_id", "pathway_name"])
    names_df["pathway_short"] = names_df["pathway_id"].str.replace("path:", "", regex=False)
    pathway_name_map = dict(zip(names_df["pathway_short"], names_df["pathway_name"]))
    print(f"Loaded {len(pathway_name_map)} pathway names")

Using cached KEGG pathway-gene links: tmp/kegg_eco_pathway_genes.tsv
Using cached KEGG pathway names: tmp/kegg_eco_pathway_names.tsv
KEGG: 137 pathways, 1785 background genes
Study genes matched: 206 / 307
Loaded 137 pathway names


In [7]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

kegg_enrich_df = pd.DataFrame()

if not kegg_link_df.empty and len(kegg_study_genes) > 0:
    N = len(kegg_background)
    K = len(kegg_study_genes)

    rows = []
    for pw, grp in kegg_link_df.groupby("pathway_short"):
        pw_genes = set(grp["locus_id"])
        a = len(kegg_study_genes & pw_genes)  # study in pathway
        b = len(pw_genes) - a                  # pop-only in pathway
        c = K - a                               # study not in pathway
        d = N - K - b                           # pop-only, not in pathway
        if a == 0:
            continue
        _, pval = fisher_exact([[a, b], [c, max(d, 0)]], alternative="greater")
        rows.append({
            "pathway_id":       pw,
            "pathway_name":     pathway_name_map.get(pw, pw),
            "study_in_pathway": a,
            "pathway_size":     len(pw_genes),
            "p_value":          pval,
        })

    kegg_enrich_df = pd.DataFrame(rows)
    if not kegg_enrich_df.empty:
        _, fdr, _, _ = multipletests(kegg_enrich_df["p_value"], method="fdr_bh")
        kegg_enrich_df["p_fdr"] = fdr
        kegg_enrich_df = kegg_enrich_df.sort_values("p_fdr")
        sig_k = kegg_enrich_df[kegg_enrich_df["p_fdr"] < 0.05]
        print(f"KEGG pathways tested: {len(kegg_enrich_df)}, significant (FDR<0.05): {len(sig_k)}")
        print("\nTop 15 KEGG pathways:")
        print(kegg_enrich_df.head(15)[
            ["pathway_name", "study_in_pathway", "pathway_size", "p_fdr"]
        ].to_string(index=False))
else:
    print("Skipping KEGG enrichment: no data.")

KEGG pathways tested: 70, significant (FDR<0.05): 13

Top 15 KEGG pathways:
                                                          pathway_name  study_in_pathway  pathway_size    p_fdr
        Phosphotransferase system (PTS) - Escherichia coli K-12 MG1655                19            43 0.000003
 Phosphonate and phosphinate metabolism - Escherichia coli K-12 MG1655                 7             8 0.000042
                       ABC transporters - Escherichia coli K-12 MG1655                42           179 0.000042
      Degradation of aromatic compounds - Escherichia coli K-12 MG1655                10            17 0.000056
         Exopolysaccharide biosynthesis - Escherichia coli K-12 MG1655                 8            15 0.001212
                    Folate biosynthesis - Escherichia coli K-12 MG1655                10            26 0.003388
               Nitrotoluene degradation - Escherichia coli K-12 MG1655                 5             7 0.003388
      Ascorbate and aldarate